# ProtCell protein imputation

Fine-tune ProtCell on training cells, reconstruct masked measured proteins, and inspect protein-wise recovery and abundance errors.

Run the cells in order with the `protein_llm` kernel. The setup cell uses the bundled data directory and `checkpoints/protcell_pretrained.pt`. The `RUN_*` switches enable the corresponding fine-tuning, prediction or analysis steps.

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

PROJECT_ROOT = Path(os.environ.get("PROTCELL_PROJECT_ROOT", str(Path.cwd()))).expanduser().resolve()
if not (PROJECT_ROOT / "train.py").is_file() and (PROJECT_ROOT.parent / "train.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "train.py").is_file(), "Set PROTCELL_PROJECT_ROOT to your ProtCell clone before starting JupyterLab."
OUTPUT_ROOT = Path(os.environ.get("PROTCELL_OUTPUT_DIR", str(PROJECT_ROOT / "results/outputs")))
OUTPUT_DIR = OUTPUT_ROOT / "imputation"
FIGURE_DIR = OUTPUT_DIR / "figures"
RESULT_DIR = OUTPUT_DIR / "results"
MODEL_DIR = OUTPUT_DIR / "models"
for directory in [OUTPUT_DIR, FIGURE_DIR, RESULT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.chdir(OUTPUT_DIR) # trainer logs and new files stay with this experiment

old_cache_settings = {key: os.environ.get(key) for key in ["MPLCONFIGDIR", "NUMBA_CACHE_DIR", "XDG_CACHE_HOME"]}
plot_cache = tempfile.TemporaryDirectory(prefix="protcell-imputation-", dir=OUTPUT_DIR)
os.environ["MPLCONFIGDIR"] = plot_cache.name
os.environ["NUMBA_CACHE_DIR"] = str(Path(plot_cache.name) / "numba")
os.environ["XDG_CACHE_HOME"] = str(Path(plot_cache.name) / "xdg")

import json
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse, stats
from tqdm.auto import tqdm
from IPython.display import display
from model.model import ProteinLanguageModel
from typing import List
from finetuner.Imputation import ImputationTrainer
from scipy.spatial.distance import cosine
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.family": "sans-serif"})
sc.settings.figdir = FIGURE_DIR
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DATA_ROOT = Path(os.environ.get("PROTCELL_DATA_ROOT", str(PROJECT_ROOT / "results/data"))).expanduser().resolve()
BASE_MODEL_PATH = PROJECT_ROOT / "checkpoints/protcell_pretrained.pt"
print("Device:", DEVICE)
print("Outputs:", OUTPUT_DIR)

## Choose the dataset and split

Choose a prepared example and define training and test cells. The existing half-cell examples use consecutive halves; the cross-batch examples select cells using their batch or condition labels.

In [ ]:
EXPERIMENTS = {
    "119-halfcells": ("119", "ARCSIN", None, None),
    "34-halfcells": ("34", "ARCSIN", None, None),
    "83_halfcells": ("83", "CLR", None, None),
    "119_1": ("119", "ARCSIN", "condition", "1"),
    "83_batch1": ("83", "CLR", "batch", "1"),
}
EXPERIMENT = "83_batch1"
dataset_id, transform, split_column, train_value = EXPERIMENTS[EXPERIMENT]
adata = ad.read_h5ad(DATA_ROOT / f"{transform}_data/Dataset{dataset_id}.h5ad")
if split_column is None:
    train_data = adata[:adata.n_obs // 2].copy()
    test_data = adata[adata.n_obs // 2:].copy()
else:
    is_train = adata.obs[split_column].astype(str).eq(train_value)
    train_data = adata[is_train].copy()
    test_data = adata[~is_train].copy()
assert set(train_data.obs_names).isdisjoint(test_data.obs_names)
display(train_data); display(test_data)
RUN_FINETUNING = True
RUN_QUARTER_TEST = True
RUN_SINGLE_PROTEIN_TEST = True
IMPUTATION_MODEL_PATH = PROJECT_ROOT / f"checkpoints/imputation/{EXPERIMENT}/base_10.pt"

## Load the pretrained backbone

Use this object for a new fine-tuning run. The saved experiment checkpoint is loaded separately for evaluation.

In [ ]:
model = ProteinLanguageModel.load_model_from_checkpoint(str(BASE_MODEL_PATH), device=DEVICE).to(DEVICE)
display(model)

## Fine-tune on training cells

Fine-tune ProtCell with ImputationTrainer using the parameters below.


In [ ]:
if RUN_FINETUNING:
    trainer = ImputationTrainer(
        model=model, adata=train_data, device=DEVICE, batch_size=512,
        expr_name=EXPERIMENT, lr=1e-3, train_cls=False,
        trainable_modules={"lora": 1e-3, "reg_proj": 1e-5, "expr_head": 5e-5},
    )
    trainer.fit(num_epochs=10, save=True, save_dir=str(MODEL_DIR), save_epoch_interval=10,
                lora_rank=16, lora_alpha=64, lora_dropout=0.1)
    IMPUTATION_MODEL_PATH = MODEL_DIR / EXPERIMENT / "base_10.pt"

## Load the imputation model

The checkpoint contains the dataset-adapted ProtCell model.

In [ ]:
imputation_model = ProteinLanguageModel.load_model_from_checkpoint(str(IMPUTATION_MODEL_PATH), device=DEVICE).to(DEVICE)
imputation_model.eval()
display(imputation_model)

## Masked forward pass

Predict masked protein abundances in batches and return predictions in the order specified by predict_protein.

In [ ]:
def predict_values(model, 
                        adata, 
                        predict_protein: List,
                        device='cuda',
                        batch_size=2048,
                        show_tqdm = True,
                        ):

    assert len(predict_protein) > 0, "predict_protein cannot be empty"
    assert len(set(predict_protein)) == len(predict_protein), \
        "predict_protein has duplicate protein names"
    assert adata.var_names.is_unique, "adata.var_names has duplicate protein names"

    model.eval()
    model.to(device)

    # generate mask and add protein
    missing = [p for p in predict_protein if p not in adata.var_names]

    X = adata.X.toarray() if sparse.issparse(adata.X) else adata.X

    if len(missing) > 0:
        print(f"补充缺失的蛋白列: {missing}")
        added_values = np.zeros(shape=(X.shape[0], len(missing)))

        X = np.hstack((X, added_values))

        var_old = adata.var.copy()
        var_new = pd.DataFrame(index=missing)
        var_final = pd.concat([var_old, var_new])

        adata = ad.AnnData(
            X=X,
            obs=adata.obs.copy(),
            var=var_final
        )

    input_ids = np.array(model.tokenizer.encode(adata.var_names.to_list()))
    input_ids = torch.tensor(input_ids).to(device)
    input_ids = input_ids.expand(adata.n_obs, -1)
    expr_values = torch.tensor(X).float().to(device)

    var_names = np.array(adata.var_names.tolist())
    mask_local = np.isin(var_names, np.array(predict_protein))

    # 模型按矩阵列顺序返回被 mask 的蛋白，
    # 记录如何将输出重排为 predict_protein 的顺序。
    masked_proteins = var_names[mask_local].tolist()
    output_order = [masked_proteins.index(p) for p in predict_protein]

    expr_preds_all = []
    a = tqdm(range(0, adata.n_obs, batch_size), desc="Predict Protein Values") if show_tqdm else range(0, adata.n_obs, batch_size)
    for i in a:
        batch_expr = expr_values[i : i + batch_size, :].float()
        batch_ids = input_ids[i : i + batch_size, :]
        B, L = batch_expr.shape
        mask = np.tile(mask_local, (B, 1))
        mask = torch.tensor(mask, dtype=torch.bool, device=device)

        with torch.no_grad():
            expr_preds, _ = model(input_ids=batch_ids, expr_values=batch_expr, mask=mask)
            expr_preds_all.append(expr_preds.cpu())

    predictions = (
        torch.cat(expr_preds_all, dim=0)
        .reshape(adata.n_obs, len(masked_proteins))
        .numpy()
    )

    return predictions[:, output_order]

## Protein-wise metric calculation

Compute correlations, cosine similarity and abundance errors across test cells for each protein. Normalize MSE by the target variance and MAE by its standard deviation.

In [ ]:
def protein_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mse = np.mean((y_true - y_pred) ** 2)
    mae = np.mean(np.abs(y_true - y_pred))
    scale = np.std(y_true)
    return {
        "pearson": stats.pearsonr(y_true, y_pred).statistic,
        "spearman": stats.spearmanr(y_true, y_pred).statistic,
        "cosine": 1 - cosine(y_true, y_pred) if np.linalg.norm(y_true) and np.linalg.norm(y_pred) else 0.0,
        "mse": mse, "mae": mae,
        "nmse": mse / scale ** 2 if scale > 0 else np.nan,
        "nmae": mae / scale if scale > 0 else np.nan,
    }

## Test simultaneous quarter-protein masking

The target identities are specified before prediction and retained in the output table.

In [ ]:
quarter_metrics = None
quarter_targets = test_data.var_names[:test_data.n_vars // 4].tolist()
if RUN_QUARTER_TEST:
    quarter_prediction = predict_values(imputation_model, test_data, quarter_targets, device=DEVICE, batch_size=512)
    truth = test_data[:, quarter_targets].X
    truth = truth.toarray() if sparse.issparse(truth) else np.asarray(truth)
    quarter_metrics = pd.DataFrame([
        {"protein": protein, **protein_metrics(truth[:, j], quarter_prediction[:, j])}
        for j, protein in enumerate(quarter_targets)
    ])
    quarter_metrics.to_csv(RESULT_DIR / f"new_{EXPERIMENT}_quarter_metrics.csv", index=False)
    np.savez_compressed(RESULT_DIR / f"new_{EXPERIMENT}_quarter_predictions.npz",
                        cell_ids=test_data.obs_names.to_numpy(), proteins=np.array(quarter_targets),
                        truth=truth, prediction=quarter_prediction)
    display(quarter_metrics)

## Test one protein at a time

Each measured protein is masked independently while the remaining measured proteins provide context.

In [ ]:
single_metrics = None
if RUN_SINGLE_PROTEIN_TEST:
    single_rows = []
    for protein in tqdm(test_data.var_names, desc="Single-protein masking"):
        prediction = predict_values(imputation_model, test_data, [protein], device=DEVICE,
                                    batch_size=512, show_tqdm=False)[:, 0]
        truth = test_data[:, [protein]].X
        truth = truth.toarray().ravel() if sparse.issparse(truth) else np.asarray(truth).ravel()
        single_rows.append({"protein": protein, **protein_metrics(truth, prediction)})
    single_metrics = pd.DataFrame(single_rows)
    single_metrics.to_csv(RESULT_DIR / f"new_{EXPERIMENT}_single_metrics.csv", index=False)
    display(single_metrics)

## Display the protein-wise results

Use the tables calculated by this run's simultaneous and single-protein masking cells. CSV files are exported for sharing.


In [ ]:
protein_tables = {}
for mode, enabled, metrics in [
    ("quarter", RUN_QUARTER_TEST, quarter_metrics),
    ("single", RUN_SINGLE_PROTEIN_TEST, single_metrics),
]:
    if not enabled:
        continue
    assert metrics is not None, f"Run the {mode} prediction cell before displaying results."
    frame = metrics.copy()
    assert frame.protein.is_unique
    protein_tables[mode] = frame
    display(frame)


## Plot masked-protein recovery

Display ProtCell’s protein-wise correlations and abundance errors for simultaneous masking.

In [ ]:
if "quarter" in protein_tables:
    quarter_results = protein_tables["quarter"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    quarter_results.set_index("protein")[["pearson", "spearman", "cosine"]].plot.bar(ax=axes[0])
    quarter_results.set_index("protein")[["mse", "mae"]].plot.bar(ax=axes[1])
    axes[0].set_title("ProtCell masked-protein recovery"); axes[0].set_ylabel("Similarity")
    axes[1].set_title("ProtCell abundance errors"); axes[1].set_ylabel("Error")
    fig.tight_layout(); fig.savefig(FIGURE_DIR / "ProtCell_quarter_recovery.pdf", bbox_inches="tight"); plt.show()
    display(quarter_results.drop(columns="protein").agg(["mean", "median"]))

## Plot one-protein-at-a-time recovery

Inspect which proteins are recovered well when their own measured abundances are hidden.

In [ ]:
if "single" in protein_tables:
    single_results = protein_tables["single"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    similarities = single_results.melt(id_vars="protein", value_vars=["pearson", "spearman", "cosine"],
                                       var_name="metric", value_name="value")
    sns.violinplot(data=similarities, x="metric", y="value", cut=0, inner="quartile", ax=axes[0])
    strongest = single_results.sort_values("pearson", ascending=False)
    axes[1].scatter(np.arange(len(strongest)), strongest.pearson, s=20, color="#659AAF")
    axes[1].set_xlabel("Proteins ordered by Pearson correlation"); axes[1].set_ylabel("Pearson correlation")
    fig.tight_layout(); fig.savefig(FIGURE_DIR / "ProtCell_single_protein_recovery.pdf", bbox_inches="tight"); plt.show()
    display(strongest[["protein", "pearson", "spearman", "mse", "mae"]])

## Release temporary files

Keep the final tables, figures and trained models; delete this notebook’s temporary plotting/JIT caches.

In [ ]:
plt.close("all")
plot_cache.cleanup()
for key, previous in old_cache_settings.items():
    if previous is None:
        os.environ.pop(key, None)
    else:
        os.environ[key] = previous
print("Temporary caches removed.")